<div align="center">

<a href="https://colab.research.google.com/github/utkukose/rd-project-management-NB-lecture/blob/main/weeks/week-11/NB11_agile_research.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 11: Agile Methods in Research and Development

**R&D and Project Management in Computer Science (11117BLG002)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/rd-project-management-NB-lecture/weeks/week-11/lab.html>.

## Overview

Plans made at the start of a research project rarely survive contact with results. This week introduces the values of agile development, the Scrum framework, Kanban and the law that links work in progress to cycle time, and discusses how iterative methods can work inside grant agreements with fixed deliverables [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to explain the values and principles of agile development, to describe the accountabilities, events and artefacts of Scrum, to apply Little's law to a team's flow of work, to forecast delivery from past velocity, and to design a hybrid approach for a funded research project.

## Agile values

In 2001, a group of software practitioners published the Manifesto for Agile Software Development [1]. It values individuals and interactions over processes and tools, working software over comprehensive documentation, customer collaboration over contract negotiation, and responding to change over following a plan, while recognising value in the items on the right. Twelve principles elaborate these values, among them the frequent delivery of working software and regular reflection on how to become more effective. Dingsøyr and colleagues reviewed a decade of research on agile methods and argued that the field needed stronger theoretical foundations to explain why and when the methods work [4].

## Scrum

The Scrum Guide describes Scrum as a lightweight framework for addressing complex problems [2]. A Scrum Team has three accountabilities: Developers, who create the increment, a Product Owner, who maximises the value of the product and manages the Product Backlog, and a Scrum Master, who helps the team use Scrum effectively. Work proceeds in Sprints of one month or less, each with Sprint Planning, a Daily Scrum, a Sprint Review and a Sprint Retrospective. The three artefacts, the Product Backlog, the Sprint Backlog and the Increment, carry commitments: the Product Goal, the Sprint Goal and the Definition of Done. The velocity of past Sprints supports forecasts, which the lab turns into probability distributions.

## Flow and Little's law

Kanban, as described by Anderson, makes work visible on a board, limits the work in progress and manages the flow of items through the system [5]. Little proved that, in a stable system, the average number of items in the system equals the arrival rate times the average time an item spends in it [3]. For a team, this means that the average cycle time equals the average work in progress divided by the throughput. If throughput is limited by the team's capacity, starting more work does not finish more work; it only makes each item take longer, as Figure 11.1 shows. Researchers who work on many papers and tasks at once experience the same effect.

![Figure 11.1](https://raw.githubusercontent.com/utkukose/rd-project-management-NB-lecture/main/weeks/week-11/figures/w11_fig1.png)

*Figure 11.1. Simulated Kanban system with fixed capacity: raising the limit on work in progress leaves throughput unchanged and lengthens cycle times, as Little's law predicts [3].*

## Agile inside a research grant

Research projects combine a fixed structure, with work packages, deliverables and milestones agreed with the funder, and uncertain content, since results determine the next steps. Marchesi and colleagues described the use of a distributed form of Scrum to manage a European research project that developed an agent-based software platform, arguing that research projects are complex and must be adapted continuously [6]. A common hybrid keeps the work packages and milestones of the grant as the stable frame and runs iterations inside them: a backlog of hypotheses, experiments and software tasks, short cycles with reviews against the objectives, and limits on work in progress. Changes that affect deliverables still follow the funder's amendment rules.

> **Pause and reflect.** How many research tasks do you have in progress now? What does Little's law predict about their completion?

# Hands-on lab

The notebook simulates a Kanban system to check Little's law and forecasts the number of Sprints needed for a backlog by resampling past velocities [2, 3].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A Kanban simulation

In [ ]:
def kanban(limit, arrival=1.5, capacity=3.0, days=400, warmup=100, seed=SEED):
    rng = np.random.default_rng(seed)
    queue, wip, cycle, in_progress, completed = [], [], [], [], 0
    for day in range(days):
        queue += [[float(rng.integers(1, 6)), None] for _ in range(rng.poisson(arrival))]
        while len(wip) < limit and queue:
            item = queue.pop(0); item[1] = day; wip.append(item)
        if day >= warmup:
            in_progress.append(len(wip))
        if wip:
            share = capacity / len(wip)
            for item in wip:
                item[0] -= share
        finished = [it for it in wip if it[0] <= 1e-9]
        wip = [it for it in wip if it[0] > 1e-9]
        if day >= warmup:
            completed += len(finished)
            cycle += [day + 1 - it[1] for it in finished if it[1] >= warmup]
    return np.mean(in_progress), completed / (days - warmup), np.mean(cycle)

for limit in [2, 4, 8]:
    L, lam, W = kanban(limit)
    print(f"limit {limit}: L = {L:.2f}, throughput = {lam:.2f}, W = {W:.2f}, L / throughput = {L / lam:.2f}")

### Your turn, exercise 1

A research group has on average 6 manuscripts in progress and completes 1.5 manuscripts per month. Use Little's law to compute the average time, in months, that a manuscript spends in progress.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _little_reference():
    return 6 / 1.5

In [ ]:
time_in_progress = None  # your computation
check("Exercise 1", time_in_progress, _little_reference())

## 2. Forecasting from velocity

In [ ]:
VELOCITIES = np.array([21, 18, 25, 17, 23, 19])

def sprints_needed(backlog, n=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        done, k = 0, 0
        while done < backlog:
            done += rng.choice(VELOCITIES); k += 1
        out.append(k)
    return np.array(out)

### Your turn, exercise 2

Compute the 85th percentile of the number of Sprints needed for a backlog of 120 points with `sprints_needed(120)`. Explain in two sentences why a forecast at the 85th percentile is more useful to a funder than one based on the average velocity.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _p85_reference():
    return float(np.percentile(sprints_needed(120), 85))

In [ ]:
p85_sprints = None  # your computation
check("Exercise 2", p85_sprints, _p85_reference())

## Visual exploration: A cumulative flow diagram

A simulated Kanban board with a limit of four items in progress runs for sixty days. The bands show items done, in progress and waiting, and the printout checks Little's law. Change the limit and the arrival rate.

In [ ]:
rng = np.random.default_rng(SEED)
days, wip_limit = 60, 4
todo, doing, done, series = 30, 0, 0, []
for d in range(days):
    todo += rng.poisson(0.8)
    start = min(todo, max(wip_limit - doing, 0)); todo -= start; doing += start
    finish = rng.binomial(doing, 0.25); doing -= finish; done += finish
    series.append((done, doing, todo))
s = np.array(series)
plt.stackplot(range(days), s[:, 0], s[:, 1], s[:, 2], labels=["done", "in progress", "to do"], colors=["tab:green", "tab:orange", "lightgrey"])
plt.xlabel("day"); plt.ylabel("items"); plt.legend(loc="upper left"); plt.title("Cumulative flow diagram of a Kanban board"); plt.show()
throughput, wip = s[-1, 0] / days, s[:, 1].mean()
print(f"throughput {throughput:.2f} items per day, mean WIP {wip:.2f}, cycle time by Little's law {wip / throughput:.1f} days")

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/rd-project-management-NB-lecture/weeks/week-11/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which of your current tasks would you stop or pause to reduce your work in progress, and what would you expect to happen?
2. What would a Definition of Done look like for an experiment or a paper in your group?
3. Where does a grant agreement constrain agile work most in your experience, and how could the frame be designed to leave room for learning?

## Weekly task and submission

Design the working method of your project: the stable frame from the grant, the backlog, the iteration length, the limit on work in progress and the Definition of Done for experiments, software and papers. Include a velocity-based forecast for one work package and attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Agile methods under fixed contracts and grants.** Review studies of agile methods in settings with fixed scope or funding agreements and derive recommendations for research projects [4, 6].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Beck, K., Beedle, M., van Bennekum, A., et al. (2001). *Manifesto for Agile Software Development*. <https://agilemanifesto.org>

[2] Schwaber, K., & Sutherland, J. (2020). *The Scrum Guide: The Definitive Guide to Scrum: The Rules of the Game*. <https://scrumguides.org>

[3] Little, J. D. C. (1961). A proof for the queuing formula: L = λW. *Operations Research*, 9(3), 383-387. <https://doi.org/10.1287/opre.9.3.383>

[4] Dingsøyr, T., Nerur, S., Balijepally, V., & Moe, N. B. (2012). A decade of agile methodologies: Towards explaining agile software development. *Journal of Systems and Software*, 85(6), 1213-1221. <https://doi.org/10.1016/j.jss.2012.02.033>

[5] Anderson, D. J. (2010). *Kanban: Successful Evolutionary Change for Your Technology Business*. Blue Hole Press.

[6] Marchesi, M., Mannaro, K., Uras, S., & Locci, M. (2007). Distributed Scrum in research project management. In *Agile Processes in Software Engineering and Extreme Programming (XP 2007), Lecture Notes in Computer Science 4536* (pp. 240-244). <https://doi.org/10.1007/978-3-540-73101-6_45>